In [1]:
# ============================================================
# CELL 1 — LOAD GITHUB REPOSITORY
# ============================================================

import os

REPO_DIR = "/content/TA"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone https://github.com/shefiramarizcha62-sudo/TA.git /content/TA
else:
    !cd /content/TA && git pull origin main

print("=" * 70)
print("REPOSITORY READY")
print("=" * 70)

print(REPO_DIR)

Cloning into '/content/TA'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 54 (delta 10), reused 40 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 1.24 MiB | 15.53 MiB/s, done.
Resolving deltas: 100% (10/10), done.
REPOSITORY READY
/content/TA


In [2]:
# ============================================================
# CELL 2 — LOAD REVISED DATASET
# ============================================================

import pandas as pd
import os

DATASET_PATH = os.path.join(
    REPO_DIR,
    "dataset",
    "dataset_preprocessed_C0-C4_revised.csv"
)

df = pd.read_csv(DATASET_PATH)

print("=" * 70)
print("REVISED DATASET")
print("=" * 70)

print("Path :", DATASET_PATH)
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

REVISED DATASET
Path : /content/TA/dataset/dataset_preprocessed_C0-C4_revised.csv
Shape: (3815, 13)

Columns:
['No', 'Komentar', 'Label Awal', 'Label Pakar', 'Pasal Dominan', 'Catatan', 'Unnamed: 6', 'Kode Label', 'jumlah_kata', 'jumlah_karakter', 'clean_comment', 'Komentar_clean', 'Komentar_preprocessed']


,No,Komentar,Label Awal,Label Pakar,Pasal Dominan,Catatan,Unnamed: 6,Kode Label,jumlah_kata,jumlah_karakter,clean_comment,Komentar_clean,Komentar_preprocessed
0,5708.0,Apakah seorang maling anak'y jg akan menjadi m...,Indikasi penghinaan atau kebencian terhadap in...,Ambigu atau konteks tidak cukup,NaN,Pertanyaan umum tanpa target yang jelas.,NaN,C4,8.0,56.0,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anaknya juga akan menjad...
1,495.0,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...",Netral atau tidak terindikasi,Konten ofensif,NaN,Tidak terindikasi Pasal 27A maupun Pasal 28 ay...,NaN,C2,12.0,91.0,"​@hanafanda-wx1cciyaa betul, meni hese ngesahk...","​-wx1cciyaa betul, meni hese ngesahkeun pearam...","​-wx1cciyaa betul, meni hese ngesahkeun pearam..."
2,1841.0,Omongan nya pak bahlil sering kontradiksi. Lah...,Netral atau tidak terindikasi,Netral atau tidak terindikasi,NaN,NaN,NaN,C0,7.0,50.0,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...
3,6483.0,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,Indikasi penghinaan atau kebencian terhadap in...,Konten ofensif,NaN,Menuduh wakil rakyat sebagai maling dan zalim.,NaN,C2,13.0,96.0,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tetapi kel...
4,2222.0,"Cobalah pampang ke45 poin itu di media sosial,...",Netral atau tidak terindikasi,Ambigu atau konteks tidak cukup,NaN,komentar ini belum menunjukkan unsur yang cuku...,NaN,C4,12.0,76.0,"cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,..."


In [3]:
# ============================================================
# CELL 3 — VALIDASI LABEL DAN DISTRIBUSI DATASET
# ============================================================

LABEL_COL = "Kode Label"
TEXT_COL = "Komentar_preprocessed"

print("=" * 70)
print("VALIDASI DATASET")
print("=" * 70)

# Cek kolom yang dibutuhkan
required_cols = [TEXT_COL, LABEL_COL]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan!")

print("Kolom teks  :", TEXT_COL)
print("Kolom label :", LABEL_COL)

# Cek missing value
print("\nMissing value:")
print(df[required_cols].isnull().sum())

# Hapus baris yang teks/label-nya kosong
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# Pastikan label string
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Cek label unik
print("\nLabel unik:")
print(sorted(df[LABEL_COL].unique()))

# Cek distribusi label
print("\nDistribusi label:")
display(
    df[LABEL_COL]
    .value_counts()
    .sort_index()
    .rename("Jumlah")
    .to_frame()
)

print("\nTotal data:", len(df))

VALIDASI DATASET
Kolom teks  : Komentar_preprocessed
Kolom label : Kode Label

Missing value:
Komentar_preprocessed    0
Kode Label               0
dtype: int64

Label unik:
['C0', 'C1', 'C2', 'C3', 'C4']

Distribusi label:


,Jumlah
Kode Label,
C0,763
C1,763
C2,763
C3,763
C4,763



Total data: 3815


In [4]:
# ============================================================
# CELL 4 — CEK DUPLIKAT KOMENTAR LINTAS LABEL
# ============================================================

print("=" * 70)
print("CEK DUPLIKAT KOMENTAR LINTAS LABEL")
print("=" * 70)

# Hitung jumlah label berbeda untuk setiap komentar
duplicate_check = (
    df.groupby(TEXT_COL)[LABEL_COL]
    .nunique()
    .reset_index(name="jumlah_label")
)

# Komentar yang muncul pada lebih dari satu label
cross_label_duplicates = duplicate_check[
    duplicate_check["jumlah_label"] > 1
].copy()

print("Jumlah komentar yang muncul pada lebih dari satu label:",
      len(cross_label_duplicates))

if len(cross_label_duplicates) > 0:

    print("\nPERINGATAN:")
    print("Masih terdapat komentar yang muncul pada lebih dari satu label.")

    display(
        df[df[TEXT_COL].isin(cross_label_duplicates[TEXT_COL])]
        .sort_values(TEXT_COL)
        [[TEXT_COL, LABEL_COL]]
        .head(50)
    )

else:
    print("\n✓ Tidak ditemukan komentar yang sama pada label berbeda.")

CEK DUPLIKAT KOMENTAR LINTAS LABEL
Jumlah komentar yang muncul pada lebih dari satu label: 0

✓ Tidak ditemukan komentar yang sama pada label berbeda.


In [5]:
# ============================================================
# CELL 6 — PREPARE DATASET UNTUK SPLITTING
# ============================================================

from sklearn.model_selection import train_test_split

print("=" * 70)
print("PREPARE DATASET")
print("=" * 70)

# Hanya gunakan teks dan label untuk eksperimen klasifikasi
data_split = df[[TEXT_COL, LABEL_COL]].copy()

# Mapping label C0-C4 ke integer
label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

# Validasi label
unknown_labels = set(data_split[LABEL_COL]) - set(label2id.keys())

if unknown_labels:
    raise ValueError(
        f"Ditemukan label yang tidak dikenali: {unknown_labels}"
    )

data_split["label"] = data_split[LABEL_COL].map(label2id)

print("Shape:", data_split.shape)

display(data_split.head())

PREPARE DATASET
Shape: (3815, 3)


,Komentar_preprocessed,Kode Label,label
0,apakah seorang maling anaknya juga akan menjad...,C4,4
1,"​-wx1cciyaa betul, meni hese ngesahkeun pearam...",C2,2
2,omongan nya pak bahlil sering kontradiksi. lah...,C0,0
3,astagfirullah. katanya wakil rakyat tetapi kel...,C2,2
4,"cobalah pampang ke45 poin itu di media sosial,...",C4,4


In [6]:
# ============================================================
# CELL 7 — STRATIFIED SPLIT 80/10/10
# ============================================================

RANDOM_STATE = 42

# ------------------------------------------------------------
# STEP 1 — 80% TRAIN, 20% TEMPORARY
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    data_split,
    test_size=0.20,
    stratify=data_split["label"],
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# STEP 2 — TEMPORARY 50% VALIDATION, 50% TEST
# ------------------------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE
)

# Reset index
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=" * 70)
print("DATASET SPLIT 80/10/10")
print("=" * 70)

print(f"Total      : {len(data_split)}")
print(f"Train      : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

print("\nPersentase:")
print(f"Train      : {len(train_df)/len(data_split)*100:.2f}%")
print(f"Validation : {len(val_df)/len(data_split)*100:.2f}%")
print(f"Test       : {len(test_df)/len(data_split)*100:.2f}%")

DATASET SPLIT 80/10/10
Total      : 3815
Train      : 3052
Validation : 381
Test       : 382

Persentase:
Train      : 80.00%
Validation : 9.99%
Test       : 10.01%


In [7]:
# ============================================================
# CELL 8 — VALIDASI DISTRIBUSI LABEL SETIAP SPLIT
# ============================================================

print("=" * 70)
print("DISTRIBUSI LABEL SETIAP SPLIT")
print("=" * 70)

distribution = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).sort_index()

distribution.index = [
    id2label[i] for i in distribution.index
]

distribution["Total"] = distribution.sum(axis=1)

display(distribution)

DISTRIBUSI LABEL SETIAP SPLIT


,Train,Validation,Test,Total
C0,610,76,77,763
C1,611,76,76,763
C2,610,77,76,763
C3,610,76,77,763
C4,611,76,76,763


In [8]:
# ============================================================
# CELL 9 — CEK DATA LEAKAGE ANTAR SPLIT
# ============================================================

print("=" * 70)
print("CEK DATA LEAKAGE ANTAR SPLIT")
print("=" * 70)

train_texts = set(train_df[TEXT_COL])
val_texts = set(val_df[TEXT_COL])
test_texts = set(test_df[TEXT_COL])

train_val_overlap = train_texts.intersection(val_texts)
train_test_overlap = train_texts.intersection(test_texts)
val_test_overlap = val_texts.intersection(test_texts)

print("Train ∩ Validation :", len(train_val_overlap))
print("Train ∩ Test       :", len(train_test_overlap))
print("Validation ∩ Test  :", len(val_test_overlap))

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("\n✓ Tidak ditemukan data leakage antar split.")
else:
    print("\n⚠ DITEMUKAN OVERLAP ANTAR SPLIT.")

CEK DATA LEAKAGE ANTAR SPLIT
Train ∩ Validation : 0
Train ∩ Test       : 1
Validation ∩ Test  : 0

⚠ DITEMUKAN OVERLAP ANTAR SPLIT.


In [9]:
# ============================================================
# CELL 10 — SAVE REVISED SPLITS
# ============================================================

SPLIT_DIR = os.path.join(
    REPO_DIR,
    "dataset",
    "splits_revised"
)

os.makedirs(SPLIT_DIR, exist_ok=True)

train_path = os.path.join(
    SPLIT_DIR,
    "train_split_revised.csv"
)

val_path = os.path.join(
    SPLIT_DIR,
    "validation_split_revised.csv"
)

test_path = os.path.join(
    SPLIT_DIR,
    "test_split_revised.csv"
)

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("=" * 70)
print("REVISED SPLITS SAVED")
print("=" * 70)

print("Train      :", train_path)
print("Validation :", val_path)
print("Test       :", test_path)

REVISED SPLITS SAVED
Train      : /content/TA/dataset/splits_revised/train_split_revised.csv
Validation : /content/TA/dataset/splits_revised/validation_split_revised.csv
Test       : /content/TA/dataset/splits_revised/test_split_revised.csv


In [10]:
# ============================================================
# CELL 11 — FINAL SPLIT VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL SPLIT VERIFICATION")
print("=" * 70)

for name, path in {
    "TRAIN": train_path,
    "VALIDATION": val_path,
    "TEST": test_path
}.items():

    temp = pd.read_csv(path)

    print(
        f"{name:12s} | "
        f"shape={temp.shape} | "
        f"labels={sorted(temp['label'].unique())}"
    )

print("\n✓ Semua split berhasil disimpan.")

FINAL SPLIT VERIFICATION
TRAIN        | shape=(3052, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
VALIDATION   | shape=(381, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
TEST         | shape=(382, 3) | labels=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

✓ Semua split berhasil disimpan.


In [11]:
# ============================================================
# CELL 12 — IMPORT LIBRARY & CONFIGURATION
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("LIBRARY & CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

MODEL_NAME = "indobenchmark/indobert-large-p2"
MAX_LENGTH = 256

NUM_LABELS = 5

label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

print("Model      :", MODEL_NAME)
print("Max length :", MAX_LENGTH)
print("Num labels :", NUM_LABELS)
print("Seed       :", SEED)

print("\nDevice:")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

LIBRARY & CONFIGURATION
Model      : indobenchmark/indobert-large-p2
Max length : 256
Num labels : 5
Seed       : 42

Device:
Tesla T4


In [12]:
# ============================================================
# CELL 13 — LOAD REVISED DATASET SPLITS
# ============================================================

SPLIT_DIR = os.path.join(
    REPO_DIR,
    "dataset",
    "splits_revised"
)

TRAIN_PATH = os.path.join(
    SPLIT_DIR,
    "train_split_revised.csv"
)

VAL_PATH = os.path.join(
    SPLIT_DIR,
    "validation_split_revised.csv"
)

TEST_PATH = os.path.join(
    SPLIT_DIR,
    "test_split_revised.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=" * 70)
print("REVISED SPLITS LOADED")
print("=" * 70)

print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

display(train_df.head())

REVISED SPLITS LOADED
Train      : (3052, 3)
Validation : (381, 3)
Test       : (382, 3)


,Komentar_preprocessed,Kode Label,label
0,wah sudah parah kesal ini roman2 nya bang fery...,C0,0
1,rakyat khawatir ternyata prabowo sejenis dia.😅,C4,4
2,ah pak pigai sudah banyak minum susu ini 😂 dan...,C3,3
3,terus apa hubungan kewaspadaan dengan keturuna...,C0,0
4,yang bikin uu membolehkan siapa mirip pendukun...,C1,1


In [13]:
# ============================================================
# CELL 14 — PREPARE HUGGING FACE DATASET
# ============================================================

print("=" * 70)
print("PREPARE HUGGING FACE DATASET")
print("=" * 70)

# Pastikan hanya kolom yang dibutuhkan
train_data = train_df[[TEXT_COL, "label"]].copy()
val_data = val_df[[TEXT_COL, "label"]].copy()
test_data = test_df[[TEXT_COL, "label"]].copy()

# Pastikan label bertipe integer
train_data["label"] = train_data["label"].astype(int)
val_data["label"] = val_data["label"].astype(int)
test_data["label"] = test_data["label"].astype(int)

# Convert ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    train_data,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_data,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_data,
    preserve_index=False
)

print("Train dataset      :", train_dataset)
print("Validation dataset :", val_dataset)
print("Test dataset       :", test_dataset)

PREPARE HUGGING FACE DATASET
Train dataset      : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 3052
})
Validation dataset : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 381
})
Test dataset       : Dataset({
    features: ['Komentar_preprocessed', 'label'],
    num_rows: 382
})


In [14]:
# ============================================================
# CELL 15 — LOAD INDOBERT LARGE P2 TOKENIZER
# ============================================================

print("=" * 70)
print("LOAD TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")
print("Tokenizer:", tokenizer.__class__.__name__)

LOAD TOKENIZER


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded successfully.
Tokenizer: BertTokenizer


In [15]:
# ============================================================
# CELL 16 — TOKENIZATION
# ============================================================

def tokenize_function(examples):
    return tokenizer(
        examples[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train dataset"
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation dataset"
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test dataset"
)

print("=" * 70)
print("TOKENIZATION COMPLETE")
print("=" * 70)

print("Train columns:")
print(train_tokenized.column_names)

print("\nExample:")
display(train_tokenized[0])

Tokenizing train dataset:   0%|          | 0/3052 [00:00<?, ? examples/s]

Tokenizing validation dataset:   0%|          | 0/381 [00:00<?, ? examples/s]

Tokenizing test dataset:   0%|          | 0/382 [00:00<?, ? examples/s]

TOKENIZATION COMPLETE
Train columns:
['Komentar_preprocessed', 'label', 'input_ids', 'token_type_ids', 'attention_mask']

Example:


{'Komentar_preprocessed': 'wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya',
 'label': 0,
 'input_ids': [2,
  2282,
  259,
  4762,
  10531,
  92,
  18449,
  30378,
  1107,
  1719,
  6028,
  30371,
  493,
  5536,
  2194,
  5,
  4250,
  30371,
  4007,
  1107,
  3],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [16]:
# ============================================================
# CELL 18 — DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("=" * 70)
print("DATA COLLATOR READY")
print("=" * 70)

print("Dynamic padding: enabled")

DATA COLLATOR READY
Dynamic padding: enabled


In [17]:
# ============================================================
# CELL 20 — EVALUATION METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

print("=" * 70)
print("METRICS READY")
print("=" * 70)

print("Primary metric : Macro-F1")
print("Additional     : Accuracy, Macro Precision, Macro Recall")

METRICS READY
Primary metric : Macro-F1
Additional     : Accuracy, Macro Precision, Macro Recall


In [18]:

# CELL 28 — CONTEXTUAL EMBEDDING WITH INDOBERT LARGE P2
# VIEW 2

from transformers import AutoModel
from torch.utils.data import DataLoader

print("=" * 70)
print("CONTEXTUAL EMBEDDING SETUP")
print("=" * 70)

embedding_model = AutoModel.from_pretrained(
    MODEL_NAME
)

embedding_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

embedding_model.to(device)

print("Model :", MODEL_NAME)
print("Device:", device)

CONTEXTUAL EMBEDDING SETUP


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.34GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Model : indobenchmark/indobert-large-p2
Device: cuda


In [19]:
# ============================================================
# CELL 29 — MEAN POOLING FUNCTION
# ============================================================

def mean_pooling(
    model_output,
    attention_mask
):

    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    summed = torch.sum(
        token_embeddings * mask,
        dim=1
    )

    counts = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return summed / counts

In [20]:
# ============================================================
# CELL 30 — GENERATE CONTEXTUAL EMBEDDINGS
# ============================================================

def generate_embeddings(
    texts,
    batch_size=8
):

    embeddings = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            output = embedding_model(
                **encoded
            )

            pooled = mean_pooling(
                output,
                encoded["attention_mask"]
            )

        embeddings.append(
            pooled.cpu().numpy()
        )

    return np.vstack(embeddings)

In [21]:
# ============================================================
# CELL 31 — CREATE EMBEDDINGS
# ============================================================

train_embeddings = generate_embeddings(
    train_df[TEXT_COL].tolist()
)

val_embeddings = generate_embeddings(
    val_df[TEXT_COL].tolist()
)

print("=" * 70)
print("EMBEDDINGS GENERATED")
print("=" * 70)

print("Train embedding shape:", train_embeddings.shape)
print("Val embedding shape  :", val_embeddings.shape)

EMBEDDINGS GENERATED
Train embedding shape: (3052, 1024)
Val embedding shape  : (381, 1024)


In [37]:
# ============================================================
# CELL 33 — PREPARE LABELS
# ============================================================

y_train = train_df["label"].astype(int).values
y_val = val_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

print("=" * 70)
print("LABELS READY")
print("=" * 70)

print("Train:", y_train.shape)
print("Val  :", y_val.shape)
print("Test :", y_test.shape)

print("\nDistribution:")
print("Train:", np.bincount(y_train))
print("Val  :", np.bincount(y_val))
print("Test :", np.bincount(y_test))

LABELS READY
Train: (3052,)
Val  : (381,)
Test : (382,)

Distribution:
Train: [610 611 610 610 611]
Val  : [76 76 77 76 76]
Test : [77 76 76 77 76]


In [38]:
# ============================================================
# CELL 34 — EMBEDDING CLASSIFIER
# ============================================================

import torch.nn as nn

class EmbeddingClassifier(nn.Module):

    def __init__(
        self,
        input_dim=1024,
        hidden_dim=256,
        num_classes=5,
        dropout=0.2
    ):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)


embedding_classifier = EmbeddingClassifier(
    input_dim=train_embeddings.shape[1],
    hidden_dim=256,
    num_classes=NUM_LABELS,
    dropout=0.2
).to(device)

print("=" * 70)
print("EMBEDDING CLASSIFIER READY")
print("=" * 70)

print(embedding_classifier)

EMBEDDING CLASSIFIER READY
EmbeddingClassifier(
  (classifier): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=5, bias=True)
  )
)


In [39]:
# ============================================================
# CELL 35 — EMBEDDING DATALOADER
# ============================================================

from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE_EMBEDDING = 4

train_x = torch.tensor(
    train_embeddings,
    dtype=torch.float32
)

val_x = torch.tensor(
    val_embeddings,
    dtype=torch.float32
)

test_x = torch.tensor(
    test_embeddings,
    dtype=torch.float32
)

train_y = torch.tensor(
    y_train,
    dtype=torch.long
)

val_y = torch.tensor(
    y_val,
    dtype=torch.long
)

test_y = torch.tensor(
    y_test,
    dtype=torch.long
)

train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=BATCH_SIZE_EMBEDDING,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=BATCH_SIZE_EMBEDDING,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(test_x, test_y),
    batch_size=BATCH_SIZE_EMBEDDING,
    shuffle=False
)

print("=" * 70)
print("DATALOADER READY")
print("=" * 70)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

DATALOADER READY
Train batches: 763
Val batches  : 96
Test batches : 96


In [40]:
# ============================================================
# CELL 36 — CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

print("=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

for i, weight in enumerate(class_weights):
    print(f"C{i}: {weight.item():.4f}")

CLASS WEIGHTS
C0: 1.0007
C1: 0.9990
C2: 1.0007
C3: 1.0007
C4: 0.9990


In [41]:
# ============================================================
# CELL 37 — TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    embedding_classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Model          :", MODEL_NAME)
print("MAX_LENGTH     :", MAX_LENGTH)
print("Learning rate  :", LEARNING_RATE)
print("Weight decay   :", WEIGHT_DECAY)
print("Max epochs     :", MAX_EPOCHS)
print("Early stopping :", EARLY_STOPPING_PATIENCE)
print("Class weighted : True")
print("Seed           :", SEED)

#VIEW 1

TRAINING CONFIGURATION
Model          : indobenchmark/indobert-large-p2
MAX_LENGTH     : 256
Learning rate  : 2e-05
Weight decay   : 0.01
Max epochs     : 10
Early stopping : 2
Class weighted : True
Seed           : 42


In [42]:
# ============================================================
# CELL 38 — EVALUATION FUNCTION
# ============================================================

def evaluate_embedding_model(model, dataloader):

    model.eval()

    all_labels = []
    all_predictions = []

    total_loss = 0
    total_samples = 0

    with torch.no_grad():

        for batch_x, batch_y in dataloader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            total_loss += (
                loss.item() * batch_y.size(0)
            )

            total_samples += batch_y.size(0)

            all_labels.extend(
                batch_y.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    avg_loss = total_loss / total_samples

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1,
        "labels": np.array(all_labels),
        "predictions": np.array(all_predictions)
    }

In [43]:
# ============================================================
# CELL 39 — TRAIN EMBEDDING CLASSIFIER
# ============================================================

print("=" * 70)
print("TRAINING EMBEDDING CLASSIFIER")
print("=" * 70)

best_val_f1 = -np.inf
best_state = None
patience_counter = 0

training_history = []

for epoch in range(1, MAX_EPOCHS + 1):

    embedding_classifier.train()

    total_loss = 0
    total_samples = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = embedding_classifier(batch_x)

        loss = criterion(
            logits,
            batch_y
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * batch_y.size(0)
        )

        total_samples += batch_y.size(0)

    train_loss = total_loss / total_samples

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_result = evaluate_embedding_model(
        embedding_classifier,
        val_loader
    )

    val_f1 = val_result["macro_f1"]

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_result["loss"],
        "val_accuracy": val_result["accuracy"],
        "val_precision": val_result["precision"],
        "val_recall": val_result["recall"],
        "val_macro_f1": val_f1
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_result['loss']:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        best_state = {
            key: value.detach().cpu().clone()
            for key, value
            in embedding_classifier.state_dict().items()
        }

        patience_counter = 0

        print(
            f"  ✓ Best model "
            f"(Macro-F1 = {best_val_f1:.4f})"
        )

    else:

        patience_counter += 1

        print(
            f"  No improvement "
            f"({patience_counter}/{EARLY_STOPPING_PATIENCE})"
        )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if patience_counter >= EARLY_STOPPING_PATIENCE:

        print(
            f"\nEarly stopping pada epoch {epoch}."
        )

        break


# Load best validation model
embedding_classifier.load_state_dict(
    best_state
)

embedding_classifier.to(device)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Macro-F1: "
    f"{best_val_f1:.4f}"
)

TRAINING EMBEDDING CLASSIFIER
Epoch 01 | Train Loss: 1.5422 | Val Loss: 1.4665 | Val Macro-F1: 0.3991
  ✓ Best model (Macro-F1 = 0.3991)
Epoch 02 | Train Loss: 1.4434 | Val Loss: 1.3934 | Val Macro-F1: 0.4387
  ✓ Best model (Macro-F1 = 0.4387)
Epoch 03 | Train Loss: 1.3866 | Val Loss: 1.3451 | Val Macro-F1: 0.4355
  No improvement (1/2)
Epoch 04 | Train Loss: 1.3433 | Val Loss: 1.3133 | Val Macro-F1: 0.4665
  ✓ Best model (Macro-F1 = 0.4665)
Epoch 05 | Train Loss: 1.3206 | Val Loss: 1.2910 | Val Macro-F1: 0.4588
  No improvement (1/2)
Epoch 06 | Train Loss: 1.2923 | Val Loss: 1.2757 | Val Macro-F1: 0.4525
  No improvement (2/2)

Early stopping pada epoch 6.

TRAINING COMPLETE
Best Validation Macro-F1: 0.4665


In [44]:
# ============================================================
# CELL 40 — TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    training_history
)

display(history_df)

# VIEW 2

,epoch,train_loss,val_loss,val_accuracy,val_precision,val_recall,val_macro_f1
0,1,1.542173,1.466510,0.419948,0.470789,0.420267,0.399109
1,2,1.443357,1.393416,0.448819,0.442546,0.449453,0.438725
2,3,1.386580,1.345062,0.448819,0.445405,0.449419,0.435462
3,4,1.343311,1.313265,0.475066,0.466125,0.475701,0.466479
4,5,1.320562,1.290979,0.472441,0.461613,0.473172,0.458849
5,6,1.292257,1.275701,0.461942,0.458070,0.462543,0.452496


In [45]:
# ============================================================
# CELL 41 — FINAL VALIDATION
# ============================================================

val_result = evaluate_embedding_model(
    embedding_classifier,
    val_loader
)

print("=" * 70)
print("FINAL VALIDATION RESULT")
print("=" * 70)

print(f"Accuracy  : {val_result['accuracy']:.4f}")
print(f"Precision : {val_result['precision']:.4f}")
print(f"Recall    : {val_result['recall']:.4f}")
print(f"Macro-F1  : {val_result['macro_f1']:.4f}")

FINAL VALIDATION RESULT
Accuracy  : 0.4751
Precision : 0.4661
Recall    : 0.4757
Macro-F1  : 0.4665


In [46]:
# ============================================================
# CELL 42 — FINAL TEST
# ============================================================

test_result = evaluate_embedding_model(
    embedding_classifier,
    test_loader
)

print("=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)

print(f"Accuracy  : {test_result['accuracy']:.4f}")
print(f"Precision : {test_result['precision']:.4f}")
print(f"Recall    : {test_result['recall']:.4f}")
print(f"Macro-F1  : {test_result['macro_f1']:.4f}")

FINAL TEST RESULT
Accuracy  : 0.4634
Precision : 0.4605
Recall    : 0.4629
Macro-F1  : 0.4582


In [47]:
# ============================================================
# CELL 43 — CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        target_names=[
            "C0", "C1", "C2", "C3", "C4"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

          C0       0.46      0.49      0.47        77
          C1       0.46      0.43      0.45        76
          C2       0.34      0.28      0.31        76
          C3       0.48      0.62      0.54        77
          C4       0.56      0.49      0.52        76

    accuracy                           0.46       382
   macro avg       0.46      0.46      0.46       382
weighted avg       0.46      0.46      0.46       382



In [48]:
# ============================================================
# CELL 44 — CONFUSION MATRIX
# ============================================================

cm_embedding = confusion_matrix(
    test_result["labels"],
    test_result["predictions"]
)

cm_embedding_df = pd.DataFrame(
    cm_embedding,
    index=["C0", "C1", "C2", "C3", "C4"],
    columns=["C0", "C1", "C2", "C3", "C4"]
)

display(cm_embedding_df)

,C0,C1,C2,C3,C4
C0,38,12,7,8,12
C1,9,33,11,17,6
C2,19,9,21,20,7
C3,3,14,8,48,4
C4,14,3,14,8,37
